In [1]:
import pandas as pd
import numpy as np
import sqlite3
import os
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)

print("✅ Imports ready")

✅ Imports ready


In [4]:
BASE_DIR = r"C:\Users\pooja\OneDrive\Desktop\Amazon_Sales_Analytics"
EXPORTS_DIR = os.path.join(BASE_DIR, "exports")
DB_DIR      = os.path.join(BASE_DIR, "database")
os.makedirs(DB_DIR, exist_ok=True)

DB_PATH = os.path.join(DB_DIR, "amazon_india.db")
print(f"✅ Database will be created at:")
print(f"   {DB_PATH}")

✅ Database will be created at:
   C:\Users\pooja\OneDrive\Desktop\Amazon_Sales_Analytics\database\amazon_india.db


In [5]:
print("Loading cleaned transactions...")
df = pd.read_csv(
    os.path.join(EXPORTS_DIR, "amazon_transactions_clean.csv"),
    low_memory=False,
    parse_dates=['order_date']
)

print("Loading products catalog...")
catalog = pd.read_csv(
    os.path.join(EXPORTS_DIR, "amazon_products_catalog_clean.csv")
)

print(f"\n✅ Transactions : {len(df):,} rows")
print(f"✅ Products      : {len(catalog):,} rows")
print(f"\nTransaction columns: {list(df.columns)}")

Loading cleaned transactions...
Loading products catalog...

✅ Transactions : 1,127,609 rows
✅ Products      : 2,004 rows

Transaction columns: ['transaction_id', 'order_date', 'customer_id', 'product_id', 'product_name_x', 'category_x', 'subcategory_x', 'brand_x', 'original_price_inr', 'discount_percent', 'discounted_price_inr', 'quantity', 'subtotal_inr', 'delivery_charges', 'final_amount_inr', 'customer_city', 'customer_state', 'customer_tier', 'customer_spending_tier', 'customer_age_group', 'payment_method', 'delivery_days', 'delivery_type', 'is_prime_member', 'is_festival_sale', 'festival_name', 'customer_rating', 'return_status', 'order_month', 'order_year', 'order_quarter', 'product_weight_kg', 'is_prime_eligible_x', 'product_rating', 'source_year', 'product_name_y', 'category_y', 'subcategory_y', 'brand_y', 'base_price_2015', 'weight_kg', 'rating', 'is_prime_eligible_y', 'launch_year', 'model']


In [6]:
# ── 1. TIME DIMENSION ────────────────────────────────────────
print("Building time dimension...")
dates = pd.to_datetime(df['order_date'].unique())
time_dim = pd.DataFrame({
    'date_id'     : pd.to_datetime(dates).strftime('%Y%m%d').astype(int),
    'full_date'   : pd.to_datetime(dates).strftime('%Y-%m-%d'),
    'year'        : pd.to_datetime(dates).year,
    'quarter'     : pd.to_datetime(dates).quarter,
    'month'       : pd.to_datetime(dates).month,
    'month_name'  : pd.to_datetime(dates).strftime('%B'),
    'month_short' : pd.to_datetime(dates).strftime('%b'),
    'week'        : pd.to_datetime(dates).isocalendar().week.astype(int),
    'day_of_week' : pd.to_datetime(dates).dayofweek,
    'day_name'    : pd.to_datetime(dates).strftime('%A'),
    'is_weekend'  : (pd.to_datetime(dates).dayofweek >= 5).astype(int),
    'is_month_end': pd.to_datetime(dates).is_month_end.astype(int),
})
time_dim = time_dim.drop_duplicates(subset='date_id').sort_values('date_id')
print(f"  time_dim: {len(time_dim):,} rows")

# ── 2. CUSTOMER DIMENSION ──────────────────────────────────────
print("Building customer dimension...")
customers = (df.groupby('customer_id')
               .agg(
                   customer_city         = ('customer_city',          'first'),
                   customer_state        = ('customer_state',         'first'),
                   customer_tier         = ('customer_tier',          'first'),
                   customer_spending_tier= ('customer_spending_tier', 'first'),
                   customer_age_group    = ('customer_age_group',     'first'),
                   is_prime_member       = ('is_prime_member',        'max'),
                   first_purchase_year   = ('order_year',             'min'),
                   last_purchase_year    = ('order_year',             'max'),
                   total_orders          = ('transaction_id',         'count'),
                   total_spend           = ('final_amount_inr',       'sum'),
                   avg_order_value       = ('final_amount_inr',       'mean'),
               )
               .reset_index())
print(f"  customers: {len(customers):,} rows")

# ── 3. PRODUCT DIMENSION ───────────────────────────────────────
print("Building product dimension...")
# Use catalog as base, enrich with transaction-level stats
prod_stats = (df.groupby('product_id')
                .agg(
                    total_orders  = ('transaction_id', 'count'),
                    total_revenue = ('final_amount_inr','sum'),
                    avg_rating    = ('product_rating',  'mean'),
                    return_count  = ('return_status',
                                    lambda x: (x=='Returned').sum()),
                )
                .reset_index())

products = catalog.merge(prod_stats, on='product_id', how='left')
products['return_count'] = products['return_count'].fillna(0)
print(f"  products: {len(products):,} rows")

# ── 4. FACT TABLE (Transactions) ──────────────────────────────
print("Building fact table...")
df['date_id'] = pd.to_datetime(df['order_date']).dt.strftime('%Y%m%d').astype(int)

FACT_COLS = [
    'transaction_id', 'date_id', 'order_date',
    'customer_id', 'product_id',
    'category', 'subcategory', 'brand',
    'original_price_inr', 'discount_percent',
    'discounted_price_inr', 'quantity',
    'delivery_charges', 'final_amount_inr',
    'payment_method', 'delivery_days', 'delivery_type',
    'is_prime_member', 'is_festival_sale', 'festival_name',
    'customer_rating', 'product_rating', 'return_status',
    'order_year', 'order_month', 'order_quarter',
]
FACT_COLS = [c for c in FACT_COLS if c in df.columns]
fact = df[FACT_COLS].copy()
print(f"  fact (transactions): {len(fact):,} rows")

print("\n✅ All 4 tables built")

Building time dimension...
  time_dim: 4,015 rows
Building customer dimension...
  customers: 354,969 rows
Building product dimension...
  products: 2,004 rows
Building fact table...
  fact (transactions): 1,127,609 rows

✅ All 4 tables built


In [10]:
import os
import sqlite3

# ── Remove existing DB safely ─────────────────────────────
try:
    conn.close()
except NameError:
    pass  # No existing connection
except:
    pass

if os.path.exists(DB_PATH):
    os.remove(DB_PATH)
    print("Removed existing database")

# ── Connect to new DB ────────────────────────────────────
conn = sqlite3.connect(DB_PATH)
cur  = conn.cursor()

# ── Schema DDL ──────────────────────────────────────────
DDL = """
-- Time Dimension
CREATE TABLE time_dim (
    date_id      INTEGER PRIMARY KEY,
    full_date    TEXT NOT NULL,
    year         INTEGER,
    quarter      INTEGER,
    month        INTEGER,
    month_name   TEXT,
    month_short  TEXT,
    week         INTEGER,
    day_of_week  INTEGER,
    day_name     TEXT,
    is_weekend   INTEGER DEFAULT 0,
    is_month_end INTEGER DEFAULT 0
);

-- Customer Dimension
CREATE TABLE customers (
    customer_id            TEXT PRIMARY KEY,
    customer_city          TEXT,
    customer_state         TEXT,
    customer_tier          TEXT,
    customer_spending_tier TEXT,
    customer_age_group     TEXT,
    is_prime_member        INTEGER DEFAULT 0,
    first_purchase_year    INTEGER,
    last_purchase_year     INTEGER,
    total_orders           INTEGER,
    total_spend            REAL,
    avg_order_value        REAL
);

-- Product Dimension
CREATE TABLE products (
    product_id    TEXT PRIMARY KEY,
    product_name  TEXT,
    category      TEXT,
    subcategory   TEXT,
    brand         TEXT,
    base_price_2015 REAL,
    weight_kg     REAL,
    rating        REAL,
    is_prime_eligible INTEGER,
    launch_year   INTEGER,
    total_orders  INTEGER,
    total_revenue REAL,
    avg_rating    REAL,
    return_count  INTEGER
);

-- Fact Table
CREATE TABLE transactions (
    transaction_id       TEXT PRIMARY KEY,
    date_id              INTEGER,
    order_date           TEXT,
    customer_id          TEXT,
    product_id           TEXT,
    category             TEXT,
    subcategory          TEXT,
    brand                TEXT,
    original_price_inr   REAL,
    discount_percent     REAL,
    discounted_price_inr REAL,
    quantity             INTEGER,
    delivery_charges     REAL,
    final_amount_inr     REAL,
    payment_method       TEXT,
    delivery_days        REAL,
    delivery_type        TEXT,
    is_prime_member      INTEGER,
    is_festival_sale     INTEGER,
    festival_name        TEXT,
    customer_rating      REAL,
    product_rating       REAL,
    return_status        TEXT,
    order_year           INTEGER,
    order_month          INTEGER,
    order_quarter        INTEGER,
    FOREIGN KEY (date_id)     REFERENCES time_dim(date_id),
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id),
    FOREIGN KEY (product_id)  REFERENCES products(product_id)
);

-- Indexes
CREATE INDEX idx_txn_date      ON transactions(order_date);
CREATE INDEX idx_txn_year      ON transactions(order_year);
CREATE INDEX idx_txn_category  ON transactions(category);
CREATE INDEX idx_txn_customer  ON transactions(customer_id);
CREATE INDEX idx_txn_product   ON transactions(product_id);
CREATE INDEX idx_txn_payment   ON transactions(payment_method);
CREATE INDEX idx_txn_festival  ON transactions(is_festival_sale);
CREATE INDEX idx_txn_prime     ON transactions(is_prime_member);
CREATE INDEX idx_cust_tier     ON customers(customer_tier);
CREATE INDEX idx_cust_city     ON customers(customer_city);
"""

# ── Execute full DDL safely ──────────────────────────────
cur.executescript(DDL)
conn.commit()
print("✅ Schema created (4 tables + 10 indexes)\n")

# ── Load Data ───────────────────────────────────────────
print("Loading data...")
time_dim.to_sql('time_dim', conn, if_exists='replace', index=False)
print(f"  ✅ time_dim   : {len(time_dim):,} rows")

customers.to_sql('customers', conn, if_exists='replace', index=False)
print(f"  ✅ customers  : {len(customers):,} rows")

products.to_sql('products', conn, if_exists='replace', index=False)
print(f"  ✅ products   : {len(products):,} rows")

# Load transactions in chunks
chunk = 50000
for i in range(0, len(fact), chunk):
    fact.iloc[i:i+chunk].to_sql(
        'transactions', conn,
        if_exists='append' if i > 0 else 'replace',
        index=False
    )
    print(f"  Loading transactions... {min(i+chunk, len(fact)):,}/{len(fact):,}", end='\r')

conn.commit()
print(f"\n  ✅ transactions: {len(fact):,} rows")

size_mb = os.path.getsize(DB_PATH) / 1e6
print(f"\n✅ Database saved: {DB_PATH}")
print(f"   File size: {size_mb:.1f} MB")

# ── Close connection ─────────────────────────────────────
conn.close()

Removed existing database
✅ Schema created (4 tables + 10 indexes)

Loading data...
  ✅ time_dim   : 4,015 rows
  ✅ customers  : 354,969 rows
  ✅ products   : 2,004 rows
  Loading transactions... 1,127,609/1,127,609
  ✅ transactions: 1,127,609 rows

✅ Database saved: C:\Users\pooja\OneDrive\Desktop\Amazon_Sales_Analytics\database\amazon_india.db
   File size: 239.2 MB


In [11]:
conn = sqlite3.connect(DB_PATH)

q1 = """
SELECT
    order_year,
    COUNT(*)                                    AS total_orders,
    COUNT(DISTINCT customer_id)                 AS unique_customers,
    ROUND(SUM(final_amount_inr) / 10000000, 2)  AS revenue_crores,
    ROUND(AVG(final_amount_inr), 0)             AS avg_order_value,
    ROUND(SUM(final_amount_inr) / 10000000 /
          LAG(SUM(final_amount_inr) / 10000000)
          OVER (ORDER BY order_year) * 100 - 100, 1) AS yoy_growth_pct
FROM transactions
GROUP BY order_year
ORDER BY order_year;
"""

result = pd.read_sql_query(q1, conn)
print("📊 YEARLY REVENUE SUMMARY")
print("=" * 65)
print(result.to_string(index=False))

📊 YEARLY REVENUE SUMMARY
 order_year  total_orders  unique_customers  revenue_crores  avg_order_value  yoy_growth_pct
       2015         33165             11222          214.22        64,591.00             NaN
       2016         55275             26273          359.83        65,098.00           68.00
       2017         77385             38789          551.00        71,203.00           53.10
       2018         99495             49917          724.85        72,853.00           31.60
       2019        121605             61068          860.59        70,769.00           18.70
       2020        143715             71997        1,187.32        82,616.00           38.00
       2021        138187             69375        1,099.02        79,531.00           -7.40
       2022        132660             66605          853.23        64,317.00          -22.40
       2023        127132             63790          771.30        60,669.00           -9.60
       2024        121605             60947  

In [14]:
pd.read_sql_query("PRAGMA table_info(transactions);", conn)

,cid,name,type,notnull,dflt_value,pk
0,0,transaction_id,TEXT,0,None,0
1,1,date_id,INTEGER,0,None,0
2,2,order_date,TIMESTAMP,0,None,0
3,3,customer_id,TEXT,0,None,0
4,4,product_id,TEXT,0,None,0
5,5,original_price_inr,REAL,0,None,0
6,6,discount_percent,REAL,0,None,0
7,7,discounted_price_inr,REAL,0,None,0
8,8,quantity,INTEGER,0,None,0
9,9,delivery_charges,REAL,0,None,0


In [15]:
q_category = """
SELECT
    p.category,
    COUNT(t.transaction_id) AS total_orders,
    ROUND(SUM(t.final_amount_inr) / 10000000, 2) AS revenue_crores,
    ROUND(AVG(t.final_amount_inr), 0) AS avg_order_value,
    ROUND(AVG(t.product_rating), 2) AS avg_product_rating,
    ROUND(100.0 * SUM(CASE WHEN t.return_status = 'Returned' THEN 1 ELSE 0 END) / COUNT(*), 1) AS return_rate_pct
FROM transactions t
JOIN products p
    ON t.product_id = p.product_id
GROUP BY p.category
ORDER BY revenue_crores DESC;
"""

category_perf = pd.read_sql_query(q_category, conn)
print("📊 CATEGORY PERFORMANCE")
print("=" * 75)
print(category_perf.to_string(index=False))

📊 CATEGORY PERFORMANCE
   category  total_orders  revenue_crores  avg_order_value  avg_product_rating  return_rate_pct
Electronics       1127609        7,688.87        68,187.00                3.98             7.00


In [16]:
q3 = """
SELECT
    payment_method,
    SUM(CASE WHEN order_year = (SELECT MIN(order_year) FROM transactions)
             THEN 1 ELSE 0 END) AS first_year_orders,
    SUM(CASE WHEN order_year = (SELECT MAX(order_year) FROM transactions)
             THEN 1 ELSE 0 END) AS last_year_orders,
    COUNT(*)                    AS all_time_orders,
    ROUND(AVG(final_amount_inr), 0) AS avg_order_value
FROM transactions
GROUP BY payment_method
ORDER BY all_time_orders DESC;
"""

result = pd.read_sql_query(q3, conn)
print("📊 PAYMENT METHOD EVOLUTION")
print("=" * 75)
print(result.to_string(index=False))

📊 PAYMENT METHOD EVOLUTION
payment_method  first_year_orders  last_year_orders  all_time_orders  avg_order_value
           UPI                  0             46539           384228        65,530.00
           COD              24962              6236           322831        70,124.00
   Credit Card               4008              9286           172261        69,505.00
    Debit Card               2599              6225           140202        70,015.00
   Net Banking               1596              2285            64971        70,322.00
        Wallet                  0              1548            22821        68,582.00
          BNPL                  0              5266            20295        56,599.00


In [17]:
q4 = """
SELECT
    c.customer_city,
    c.customer_tier,
    COUNT(t.transaction_id)                     AS total_orders,
    COUNT(DISTINCT t.customer_id)               AS unique_customers,
    ROUND(SUM(t.final_amount_inr) / 10000000, 2) AS revenue_crores,
    ROUND(AVG(t.final_amount_inr), 0)           AS avg_order_value,
    ROUND(AVG(t.delivery_days), 1)              AS avg_delivery_days
FROM transactions t
JOIN customers c ON t.customer_id = c.customer_id
GROUP BY c.customer_city
ORDER BY revenue_crores DESC
LIMIT 10;
"""

result = pd.read_sql_query(q4, conn)
print("📊 TOP 10 CITIES BY REVENUE")
print("=" * 80)
print(result.to_string(index=False))

📊 TOP 10 CITIES BY REVENUE
customer_city customer_tier  total_orders  unique_customers  revenue_crores  avg_order_value  avg_delivery_days
       Mumbai         Metro        141325             41919        1,063.07        75,222.00               3.40
    New Delhi         Metro        123167             36802          926.10        75,191.00               3.40
    Bengaluru         Metro        102226             30546          765.85        74,917.00               3.40
      Chennai         Metro         84427             25183          639.38        75,732.00               3.40
      Kolkata         Metro         66867             20108          505.31        75,569.00               3.40
         Pune         Tier1         67703             22183          444.56        65,663.00               3.20
    Hyderabad         Metro         44573             13361          333.60        74,843.00               3.40
    Ahmedabad         Tier1         50861             16657          330.05  

In [18]:
q5 = """
SELECT
    CASE WHEN is_prime_member = 1 THEN 'Prime' ELSE 'Non-Prime' END AS member_type,
    COUNT(*)                                     AS total_orders,
    COUNT(DISTINCT customer_id)                  AS unique_customers,
    ROUND(SUM(final_amount_inr) / 10000000, 2)   AS revenue_crores,
    ROUND(AVG(final_amount_inr), 0)              AS avg_order_value,
    ROUND(AVG(product_rating), 2)                AS avg_product_rating,
    ROUND(AVG(delivery_days), 1)                 AS avg_delivery_days,
    ROUND(100.0 * SUM(CASE WHEN return_status = 'Returned'
                       THEN 1 ELSE 0 END) / COUNT(*), 1) AS return_rate_pct
FROM transactions
GROUP BY is_prime_member;
"""

result = pd.read_sql_query(q5, conn)
print("📊 PRIME VS NON-PRIME COMPARISON")
print("=" * 80)
print(result.to_string(index=False))

📊 PRIME VS NON-PRIME COMPARISON
member_type  total_orders  unique_customers  revenue_crores  avg_order_value  avg_product_rating  avg_delivery_days  return_rate_pct
  Non-Prime        698485            194307        4,332.99        62,034.00                3.97               4.30             7.30
      Prime        429124            160662        3,355.88        78,203.00                3.98               1.70             6.50


In [19]:
q6 = """
SELECT
    festival_name,
    COUNT(*)                                     AS total_orders,
    ROUND(SUM(final_amount_inr) / 10000000, 2)   AS revenue_crores,
    ROUND(AVG(final_amount_inr), 0)              AS avg_order_value,
    ROUND(AVG(discount_percent), 1)              AS avg_discount_pct
FROM transactions
WHERE is_festival_sale = 1
  AND festival_name != 'Unknown'
GROUP BY festival_name
ORDER BY revenue_crores DESC
LIMIT 10;
"""

result = pd.read_sql_query(q6, conn)
print("📊 TOP 10 FESTIVALS BY REVENUE")
print("=" * 75)
print(result.to_string(index=False))

📊 TOP 10 FESTIVALS BY REVENUE
               festival_name  total_orders  revenue_crores  avg_order_value  avg_discount_pct
              Back to School         85396          407.26        47,691.00             42.50
                 Diwali Sale         83611          395.82        47,340.00             42.50
Amazon Great Indian Festival         53062          251.99        47,490.00             42.50
                 Summer Sale         44931          211.92        47,166.00             42.50
               Holi Festival         39608          188.68        47,638.00             42.50
           Republic Day Sale         20183           96.02        47,576.00             42.60
              Valentine Sale         14106           66.53        47,166.00             42.40
                   Prime Day          8976           43.61        48,583.00             42.30


In [20]:
q7 = """
SELECT
    c.first_purchase_year           AS cohort_year,
    COUNT(DISTINCT t.customer_id)   AS customers_acquired,
    COUNT(t.transaction_id)         AS total_orders,
    ROUND(SUM(t.final_amount_inr) / 10000000, 2)  AS lifetime_revenue_cr,
    ROUND(AVG(c.total_spend), 0)    AS avg_clv
FROM transactions t
JOIN customers c ON t.customer_id = c.customer_id
GROUP BY c.first_purchase_year
ORDER BY cohort_year;
"""

result = pd.read_sql_query(q7, conn)
print("📊 CUSTOMER LIFETIME VALUE BY COHORT")
print("=" * 70)
print(result.to_string(index=False))

📊 CUSTOMER LIFETIME VALUE BY COHORT
 cohort_year  customers_acquired  total_orders  lifetime_revenue_cr    avg_clv
        2015               11222         89462               608.02 648,399.00
        2016               17022         91091               644.44 508,848.00
        2017               24063        103924               753.63 441,103.00
        2018               31249        116756               855.61 391,696.00
        2019               38278        127445               913.58 342,994.00
        2020               45193        136677             1,053.30 325,599.00
        2021               43559        124141               910.32 286,806.00
        2022               41837        110799               685.33 224,320.00
        2023               40085         96391               568.50 192,996.00
        2024               38234         82786               458.62 161,007.00
        2025               24227         48137               237.52 126,424.00


In [21]:
q8 = """
SELECT
    order_year,
    order_month,
    ROUND(SUM(final_amount_inr) / 10000000, 2)  AS monthly_revenue_cr,
    ROUND(
        SUM(SUM(final_amount_inr) / 10000000)
        OVER (PARTITION BY order_year
              ORDER BY order_month
              ROWS UNBOUNDED PRECEDING),
    2) AS ytd_revenue_cr
FROM transactions
GROUP BY order_year, order_month
ORDER BY order_year, order_month;
"""

result = pd.read_sql_query(q8, conn)
print("📊 MONTHLY REVENUE WITH YEAR-TO-DATE RUNNING TOTAL")
print("=" * 60)
print(result.to_string(index=False))

📊 MONTHLY REVENUE WITH YEAR-TO-DATE RUNNING TOTAL
 order_year  order_month  monthly_revenue_cr  ytd_revenue_cr
       2015            1               16.28           16.28
       2015            2               14.41           30.69
       2015            3               14.62           45.31
       2015            4               17.87           63.18
       2015            5               15.77           78.95
       2015            6               13.96           92.91
       2015            7               15.66          108.57
       2015            8               17.13          125.70
       2015            9               17.18          142.88
       2015           10               21.04          163.92
       2015           11               23.19          187.11
       2015           12               27.10          214.22
       2016            1               27.07           27.07
       2016            2               24.59           51.66
       2016            3           

In [23]:
q9_fixed = """
SELECT
    p.category,
    COUNT(*)                        AS total_orders,
    SUM(CASE WHEN t.return_status = 'Returned' THEN 1 ELSE 0 END) AS returned,
    ROUND(100.0 * SUM(CASE WHEN t.return_status = 'Returned' THEN 1 ELSE 0 END) / COUNT(*), 1) AS return_rate_pct,
    ROUND(AVG(t.product_rating), 2)   AS avg_product_rating,
    ROUND(AVG(t.customer_rating), 2)  AS avg_customer_rating,
    ROUND(AVG(t.discount_percent), 1) AS avg_discount_pct
FROM transactions t
JOIN products p ON t.product_id = p.product_id
GROUP BY p.category
ORDER BY return_rate_pct DESC;
"""

result = pd.read_sql_query(q9_fixed, conn)
print("📊 RETURN RATE & QUALITY BY CATEGORY")
print("=" * 80)
print(result.to_string(index=False))

📊 RETURN RATE & QUALITY BY CATEGORY
   category  total_orders  returned  return_rate_pct  avg_product_rating  avg_customer_rating  avg_discount_pct
Electronics       1127609     79125             7.00                3.98                 4.37             17.40


In [24]:
q10 = """
SELECT
    t.product_id,
    p.product_name,
    p.brand,
    p.category,
    COUNT(t.transaction_id)                      AS total_orders,
    ROUND(SUM(t.final_amount_inr) / 100000, 2)   AS revenue_lakhs,
    ROUND(AVG(t.product_rating), 2)              AS avg_rating,
    ROUND(100.0 * SUM(CASE WHEN t.return_status = 'Returned'
                        THEN 1 ELSE 0 END) / COUNT(*), 1) AS return_rate_pct
FROM transactions t
JOIN products p ON t.product_id = p.product_id
GROUP BY t.product_id
ORDER BY revenue_lakhs DESC
LIMIT 20;
"""

result = pd.read_sql_query(q10, conn)
print("📊 TOP 20 PRODUCTS BY REVENUE")
print("=" * 90)
print(result.to_string(index=False))
conn.close()

📊 TOP 20 PRODUCTS BY REVENUE
 product_id                      product_name     brand    category  total_orders  revenue_lakhs  avg_rating  return_rate_pct
PROD_000021      Samsung Galaxy S6 16GB Black   Samsung Electronics          2269       3,243.21        4.70             5.00
PROD_000003         Apple iPhone 6 64GB Black     Apple Electronics          2263       3,038.82        4.60             5.00
PROD_001671 Alienware Ultrabook 4GB RAM Black Alienware Electronics          2190       2,791.68        4.70             4.90
PROD_001570       ASUS Inspiron 4GB RAM Black      ASUS Electronics          1794       2,533.58        4.70             5.00
PROD_000216         Apple iPhone 8 64GB Black     Apple Electronics          1649       2,437.77        4.70             5.50
PROD_000123        Apple iPhone SE 32GB Black     Apple Electronics          1959       2,434.70        4.70             4.70
PROD_000053      OnePlus OnePlus 2 32GB Black   OnePlus Electronics          2235       2

In [26]:
# ── Connect to DB ─────────────────────────────────────────────
conn = sqlite3.connect(DB_PATH)

# ── Pre-aggregated queries for Power BI ───────────────────────
QUERIES = {
    "powerbi_yearly_revenue": """
        SELECT order_year,
               COUNT(*) AS total_orders,
               COUNT(DISTINCT customer_id) AS unique_customers,
               ROUND(SUM(final_amount_inr)/10000000,2) AS revenue_crores,
               ROUND(AVG(final_amount_inr),0) AS avg_order_value
        FROM transactions
        GROUP BY order_year
        ORDER BY order_year
    """,

    "powerbi_monthly_revenue": """
        SELECT order_year, order_month,
               ROUND(SUM(final_amount_inr)/10000000,2) AS revenue_crores,
               COUNT(*) AS total_orders,
               ROUND(AVG(final_amount_inr),0) AS avg_order_value
        FROM transactions
        GROUP BY order_year, order_month
        ORDER BY order_year, order_month
    """,

    "powerbi_category_summary": """
        SELECT p.category,
               COUNT(*) AS total_orders,
               ROUND(SUM(t.final_amount_inr)/10000000,2) AS revenue_crores,
               ROUND(AVG(t.final_amount_inr),0) AS avg_order_value,
               ROUND(AVG(t.product_rating),2) AS avg_rating
        FROM transactions t
        JOIN products p ON t.product_id = p.product_id
        GROUP BY p.category
        ORDER BY revenue_crores DESC
    """,

    "powerbi_city_summary": """
        SELECT c.customer_city, c.customer_state, c.customer_tier,
               COUNT(t.transaction_id) AS total_orders,
               ROUND(SUM(t.final_amount_inr)/10000000,2) AS revenue_crores,
               ROUND(AVG(t.final_amount_inr),0) AS avg_order_value
        FROM transactions t
        JOIN customers c ON t.customer_id=c.customer_id
        GROUP BY c.customer_city
        ORDER BY revenue_crores DESC
    """,

    "powerbi_payment_trend": """
        SELECT order_year, payment_method,
               COUNT(*) AS total_orders,
               ROUND(100.0*COUNT(*)/SUM(COUNT(*)) OVER (PARTITION BY order_year),1) AS market_share_pct
        FROM transactions
        GROUP BY order_year, payment_method
        ORDER BY order_year, total_orders DESC
    """,

    "powerbi_festival_summary": """
        SELECT festival_name,
               COUNT(*) AS total_orders,
               ROUND(SUM(final_amount_inr)/10000000,2) AS revenue_crores,
               ROUND(AVG(final_amount_inr),0) AS avg_order_value,
               ROUND(AVG(discount_percent),1) AS avg_discount_pct
        FROM transactions
        WHERE is_festival_sale=1 AND festival_name!='Unknown'
        GROUP BY festival_name
        ORDER BY revenue_crores DESC
    """,
}

# ── Export CSVs ───────────────────────────────────────────────
print("Exporting pre-aggregated tables for Power BI...\n")
for filename, sql in QUERIES.items():
    result = pd.read_sql_query(sql, conn)
    path = os.path.join(EXPORTS_DIR, f"{filename}.csv")
    result.to_csv(path, index=False)
    print(f"  ✅ {filename}.csv — {len(result):,} rows")

conn.close()

print(f"""
✅ All exports saved to: {EXPORTS_DIR}

FILES FOR POWER BI:
  amazon_transactions_clean.csv       ← main fact table
  amazon_products_catalog_clean.csv   ← product dimension
  powerbi_yearly_revenue.csv          ← pre-aggregated (fast)
  powerbi_monthly_revenue.csv         ← pre-aggregated (fast)
  powerbi_category_summary.csv        ← pre-aggregated (fast)
  powerbi_city_summary.csv            ← pre-aggregated (fast)
  powerbi_payment_trend.csv           ← pre-aggregated (fast)
  powerbi_festival_summary.csv        ← pre-aggregated (fast)
""")

Exporting pre-aggregated tables for Power BI...

  ✅ powerbi_yearly_revenue.csv — 11 rows
  ✅ powerbi_monthly_revenue.csv — 132 rows
  ✅ powerbi_category_summary.csv — 1 rows
  ✅ powerbi_city_summary.csv — 34 rows
  ✅ powerbi_payment_trend.csv — 64 rows
  ✅ powerbi_festival_summary.csv — 8 rows

✅ All exports saved to: C:\Users\pooja\OneDrive\Desktop\Amazon_Sales_Analytics\exports

FILES FOR POWER BI:
  amazon_transactions_clean.csv       ← main fact table
  amazon_products_catalog_clean.csv   ← product dimension
  powerbi_yearly_revenue.csv          ← pre-aggregated (fast)
  powerbi_monthly_revenue.csv         ← pre-aggregated (fast)
  powerbi_category_summary.csv        ← pre-aggregated (fast)
  powerbi_city_summary.csv            ← pre-aggregated (fast)
  powerbi_payment_trend.csv           ← pre-aggregated (fast)
  powerbi_festival_summary.csv        ← pre-aggregated (fast)



In [27]:
print("=" * 55)
print("PHASE 3 COMPLETE")
print("=" * 55)

conn = sqlite3.connect(DB_PATH)
for table in ['time_dim','customers','products','transactions']:
    count = pd.read_sql_query(f"SELECT COUNT(*) as n FROM {table}", conn).iloc[0,0]
    print(f"  {table:<15}: {count:,} rows")
conn.close()

size_mb = os.path.getsize(DB_PATH) / 1e6
print(f"\n  Database size : {size_mb:.1f} MB")
print(f"  Location      : {DB_PATH}")
print("""
Summary:
  ✅ Star schema: 1 fact + 3 dimension tables
  ✅ 10 indexes for fast querying
  ✅ 10 analytical SQL queries
  ✅ 6 pre-aggregated CSVs exported for Power BI

Next step → Open Power BI Desktop
""")

PHASE 3 COMPLETE
  time_dim       : 4,015 rows
  customers      : 354,969 rows
  products       : 2,004 rows
  transactions   : 1,127,609 rows

  Database size : 239.2 MB
  Location      : C:\Users\pooja\OneDrive\Desktop\Amazon_Sales_Analytics\database\amazon_india.db

Summary:
  ✅ Star schema: 1 fact + 3 dimension tables
  ✅ 10 indexes for fast querying
  ✅ 10 analytical SQL queries
  ✅ 6 pre-aggregated CSVs exported for Power BI

Next step → Open Power BI Desktop

